In [ ]:
# # 九曜：天墟 LF01 — 國漫配音 第 3 版：其他角色各一句試聽（Qwen3-TTS VoiceDesign，Kaggle）2026-09-30
#
# 旁白、蕭曜霖已由咖哩選定（第 2 版 s11 混音）。本版照同一做法（國漫風格提示詞、VoiceDesign 直接唸、4 個種子、混音）
# 替机遙、齊衡烈、郁岑燁、厲若楓、江祈璟、赤瞳妖將、院方執事各出代表台詞，先定聲音再配完整 26 句。
# 由 Claude 經 Kaggle API 推送執行；MP3 以 base64 印進執行紀錄供 Claude 取回。


In [ ]:
import subprocess
print(subprocess.run("pip install -q -U qwen-tts soundfile 2>&1 | tail -3; nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)


In [ ]:
import json, os
VOICES = json.loads(r'''{"C01": {"name": "机遙", "prompt": "熱血玄幻國漫（3D修仙動畫）的配音，標準普通話，專業配音演員的表演腔，情緒飽滿、咬字清楚。主角，十八歲少年，清亮乾淨、帶一點少年感的男聲，音調中等。平時沉穩內斂、話少，開口前會稍微停頓，語氣平靜帶一點不確定和好奇，但骨子裡有股不服輸的韌勁。"}, "C02": {"name": "齊衡烈", "prompt": "熱血玄幻國漫（3D修仙動畫）的配音，標準普通話，專業配音演員的表演腔，情緒飽滿、咬字清楚。二十歲熱血青年，明亮有力、胸腔共鳴足的男聲，音調中偏高，嗓門大、語速快，爽朗豪邁、愛笑、躍躍欲試，像隨時想找人打一架。"}, "C03": {"name": "郁岑燁", "prompt": "熱血玄幻國漫（3D修仙動畫）的配音，標準普通話，專業配音演員的表演腔，情緒飽滿、咬字清楚。二十歲劍修青年，清冷乾淨的男聲，音調中高、聲音偏薄偏亮，幾乎不帶氣聲。語氣淡漠、略帶不耐煩和驕傲，句子短促、句尾下壓，咬字銳利。"}, "C04": {"name": "厲若楓", "prompt": "熱血玄幻國漫（3D修仙動畫）的配音，標準普通話，專業配音演員的表演腔，情緒飽滿、咬字清楚。二十歲弓手青年，低沉柔和、帶一點氣聲的男聲，音量小、語速慢，句與句之間有明顯停頓。冷靜、專注、觀察入微，說話像在陳述事實。"}, "C06": {"name": "江祈璟", "prompt": "熱血玄幻國漫（3D修仙動畫）的配音，標準普通話，專業配音演員的表演腔，情緒飽滿、咬字清楚。二十歲槍修青年，沉穩冷峻的男聲，音調中低、聲音乾淨有力。驕傲自持、帶一點審視人的冷淡，說話簡潔果斷、不拖泥帶水，關鍵時刻語氣沉下來、有擔當。"}, "C08": {"name": "赤瞳妖將", "prompt": "熱血玄幻國漫（3D修仙動畫）的配音，標準普通話，專業配音演員的表演腔，情緒飽滿、咬字清楚。遠古妖族大將，非人的巨大存在。極低沉、渾厚、帶沙啞和輕微回聲般的顆粒感的男聲，語速極慢，每個字都很重。平靜、居高臨下、深不可測，帶著古老的威壓與一絲憐憫。"}, "N01": {"name": "院方執事", "prompt": "熱血玄幻國漫（3D修仙動畫）的配音，標準普通話，專業配音演員的表演腔，情緒飽滿、咬字清楚。學院登記處的中年執事，平穩清楚的男聲，音調中等，公事公辦、語氣平淡有禮，像在唱名點名，一字一句念得清楚。"}}''')
LINES = [["C01", "LF01-A08_机遙", "你們都看得出來？"], ["C01", "LF01-B05_机遙", "你不是一直不服我？"], ["C02", "LF01-A06_齊衡烈", "那就對了。第七室最近正缺個能打的。"], ["C03", "LF01-A06_郁岑燁", "你看都沒看他出手，怎麼知道他能打？"], ["C04", "LF01-A08_厲若楓", "別急。他的腳步有點亂。"], ["C06", "LF01-B01_江祈璟", "不是逃。是在讓路。"], ["C08", "LF01-B02_赤瞳妖將", "……人族。你們也開始聽見了嗎？"], ["N01", "LF01-A03_院方執事", "青嵐新契修，机遙。東廊，第七室。"]]
SEEDS = [11, 22, 33, 44]
OUT = "/kaggle/working/voice_guoman"; os.makedirs(OUT, exist_ok=True)


In [ ]:
# 直接用 VoiceDesign 唸台詞（不經 Base 照唸）；T4 → float32＋sdpa
import torch, soundfile as sf
from qwen_tts import Qwen3TTSModel
design = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", device_map="cuda:0", dtype=torch.float32, attn_implementation="sdpa")
for spk, name, text in LINES:
    for s in SEEDS:
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
        wavs, sr = design.generate_voice_design(text=text, language="Chinese", instruct=VOICES[spk]["prompt"])
        w = wavs[0][: int(sr * 25)]
        sf.write(f"{OUT}/{name}_s{s}.wav", w, sr)
        print(name, f"s{s}", f"{len(wavs[0])/sr:.2f}s", text)


In [ ]:
# （第 2 版不用 Base 照唸）


In [ ]:
# 乾聲＋混音版 MP3，並以 base64 印進執行紀錄（Claude 從雲端抓不到 Output 檔）
import glob, base64, shutil
MIX = "equalizer=f=110:t=q:w=1:g=5,equalizer=f=3500:t=q:w=1.5:g=2,acompressor=threshold=-20dB:ratio=4:attack=5:release=120:makeup=4,aecho=0.8:0.6:60|120:0.25|0.15,loudnorm=I=-16"
dst = "/kaggle/working/voice_guoman_tiny"; os.makedirs(dst, exist_ok=True)
for w in sorted(glob.glob(f"{OUT}/*.wav")):
    b = f"{dst}/{os.path.basename(w)[:-4]}"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", w, "-ac", "1", "-b:a", "64k", b + ".mp3"], check=True)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", w, "-af", MIX, "-ac", "1", "-b:a", "64k", b + "_混音.mp3"], check=True)
shutil.make_archive("/kaggle/working/LF01_voice_guoman_tiny", "zip", dst)
for m in sorted(glob.glob(f"{dst}/*.mp3")):
    print(f"@@MP3 {os.path.basename(m)} {base64.b64encode(open(m, 'rb').read()).decode()}")
